# Cyber Attack Detection

**Section 2 of the ML in Cyber Security assignment (25 marks)**

This notebook builds and compares two models for classifying network traffic into normal (BENIGN) traffic versus multiple specific attack types, and is designed to run in **Google Colab**: you upload the dataset CSV files yourself when prompted, no Kaggle API or credential setup needed.

1. **Classic Machine Learning:** PCA-reduced features + Random Forest (multi-class)
2. **Deep Learning:** 1D CNN over PCA-reduced features (multi-class)

**Dataset:** [CIC-IDS2017](https://www.unb.ca/cic/datasets/ids-2017.html) (Canadian Institute for Cybersecurity), specifically the widely-used "MachineLearningCSV" flow-feature export: 8 CSV files (one per working day / attack scenario), each row a network flow summarized by CICFlowMeter into 78 numeric features plus a `Label` column. Combined, the 8 files contain 2,830,743 flows across 15 classes (1 BENIGN class + 14 attack types), with severe class imbalance (BENIGN alone is about 80% of all rows, while some attack types such as Heartbleed have only a handful of samples). This imbalance, and how it affects evaluation, is discussed at the end of the notebook.

**Before running:** have the following 8 CSV files ready on your computer (these are the standard CIC-IDS2017 "MachineLearningCSV" filenames, matched exactly, including their original inconsistent capitalization and the official "Infilteration" typo):

- `Monday-WorkingHours.pcap_ISCX.csv`
- `Tuesday-WorkingHours.pcap_ISCX.csv`
- `Wednesday-workingHours.pcap_ISCX.csv`
- `Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv`
- `Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv`
- `Friday-WorkingHours-Morning.pcap_ISCX.csv`
- `Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv`
- `Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv`

You do **not** need to upload them into Colab yourself first - the data-loading cell below will open Colab's upload dialog automatically for whichever files it can't find (you can select multiple files at once in that dialog). See the "Load Dataset" section below for exactly where to put them if you'd rather upload ahead of time via the Files sidebar.

In [ ]:
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running in Google Colab:", IN_COLAB)
print("Working directory:", os.getcwd())

In [ ]:
import gc
import glob
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, precision_recall_curve, average_precision_score,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import joblib

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_DIR = "outputs"
FIGURES_DIR = os.path.join(OUTPUT_DIR, "figures")
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print("Setup complete.")

## 1. Load Dataset (Google Colab Upload)

This notebook does **not** download anything itself. It looks for each of the 8 expected CIC-IDS2017 CSV files in the most common places a file ends up after uploading it in Colab:

- `data/<filename>` (if you created a `data` folder in the Files sidebar and uploaded into it)
- `<filename>` directly in `/content` (the default location if you just drag-and-drop files into the Files sidebar root)

**For whichever files it can't find, the cell below will automatically open Colab's upload dialog** - you can select multiple files at once in that dialog, so you can upload all 8 in one go (or just the missing ones on a retry).

Note: anything uploaded this way only lives for the current Colab session/runtime - if the runtime restarts or disconnects, you'll need to upload again.

In [ ]:
EXPECTED_FILENAMES = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
]
CANDIDATE_DIRS = ["data", ".", "/content", "/content/data"]


def find_existing_files(filenames_needed):
    found = {}
    for directory in CANDIDATE_DIRS:
        if not os.path.isdir(directory):
            continue
        for fname in filenames_needed:
            if fname in found:
                continue
            candidate = os.path.join(directory, fname)
            if os.path.exists(candidate):
                found[fname] = candidate
    return found


def load_cic_ids2017():
    found = find_existing_files(EXPECTED_FILENAMES)
    missing = [f for f in EXPECTED_FILENAMES if f not in found]

    if missing and IN_COLAB:
        print(f"Missing {len(missing)} of {len(EXPECTED_FILENAMES)} expected files:")
        for fname in missing:
            print(" -", fname)
        print("Opening the upload dialog -- please select the missing CSV file(s) "
              "from your computer (you can select multiple at once)...")
        from google.colab import files
        files.upload()  # uploaded file(s) are saved into the current working directory
        found = find_existing_files(EXPECTED_FILENAMES)
        missing = [f for f in EXPECTED_FILENAMES if f not in found]

    if missing:
        raise FileNotFoundError(
            "Could not find the following CIC-IDS2017 files in "
            f"{CANDIDATE_DIRS}: {missing}.\n"
            "In Google Colab: re-run this cell to trigger the upload dialog, or manually upload "
            "the CSVs via the Files sidebar (into '/content' or a 'data' subfolder), then re-run "
            "this cell.\nSee README.md for details."
        )

    frames = []
    for fname in EXPECTED_FILENAMES:
        frame = pd.read_csv(found[fname])
        print(f"Loaded {fname}: {frame.shape}")
        frames.append(frame)

    combined = pd.concat(frames, ignore_index=True)
    del frames
    gc.collect()
    return combined


df = load_cic_ids2017()
print("\nCombined shape:", df.shape)
df.head()

## 2. Data Cleaning

The raw CIC-IDS2017 CSVs have several well-known quirks that need handling before any modeling:

1. **Column name whitespace** - most column names have a leading space (e.g. `" Label"`), inconsistently, because of how CICFlowMeter exported them.
2. **A duplicated column** - `Fwd Header Length` appears twice in the raw header; pandas automatically renames the second occurrence to `Fwd Header Length.1` when reading the CSV, so both survive as distinct (redundant) features rather than colliding.
3. **A corrupted character in some Web Attack labels** - the original dataset publisher's own encoding pass replaced the dash in labels like `Web Attack - Brute Force` with the Unicode replacement character (`�`), permanently, at the source. This is cleaned up by replacing it with a plain hyphen.
4. **Infinite and missing values** - columns like `Flow Bytes/s` and `Flow Packets/s` contain literal `Infinity` and `NaN` values for flows with zero duration. These are converted to proper `NaN` and then dropped.
5. **No categorical predictor columns** - this particular CSV export only contains the 78 numeric flow-based features plus the `Label` column (no `Protocol`, IP address, or timestamp columns), so there is nothing else to encode other than the target label itself.

In [ ]:
df.columns = df.columns.str.strip()

df["Label"] = df["Label"].str.strip().str.replace("�", "-", regex=False)

print("Rows before cleaning:", len(df))
print("Distinct labels:", df["Label"].nunique())

feature_cols = [c for c in df.columns if c != "Label"]
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)

n_missing = df[feature_cols].isna().any(axis=1).sum()
print(f"Rows with missing/infinite values: {n_missing}")

df = df.dropna(subset=feature_cols).reset_index(drop=True)
df[feature_cols] = df[feature_cols].astype("float32")

gc.collect()

print("Rows after cleaning:", len(df))
print("\nClass distribution after cleaning:")
print(df["Label"].value_counts())

## 3. Exploratory Data Analysis

The class distribution is heavily imbalanced (BENIGN traffic dominates), so it is plotted on a log scale to make the rarer attack classes visible at all.

In [ ]:
class_counts = df["Label"].value_counts()

plt.figure(figsize=(9, 5))
sns.barplot(x=class_counts.values, y=class_counts.index, orient="h")
plt.xscale("log")
plt.xlabel("Number of flows (log scale)")
plt.ylabel("Class")
plt.title("Class Distribution (CIC-IDS2017)")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "class_distribution.png"), dpi=150)
plt.show()

print(class_counts)

## 4. Label Encoding, Train/Test Split, and Standardization

The multi-class target (`Label`, 15 classes) is integer-encoded with `LabelEncoder`. An 80/20 stratified split keeps every class's proportion the same in train and test. `StandardScaler` is fit only on the training set and then applied to both, to avoid leaking test-set statistics into preprocessing.

Some attack classes have very few samples in total (as few as 11 for Heartbleed), so their test-set counts will be tiny; this is a genuine limitation of the dataset, discussed later, not a bug.

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["Label"])
X = df[feature_cols].to_numpy(dtype="float32")

CLASS_NAMES = label_encoder.classes_
NUM_CLASSES = len(CLASS_NAMES)
print(f"Number of classes: {NUM_CLASSES}")
print("Classes:", list(CLASS_NAMES))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"\nTrain size: {len(X_train)}, Test size: {len(X_test)}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

del X, X_train, X_test
gc.collect()

## 5. Feature Selection with PCA

`PCA` is fit on the standardized training features only, keeping enough principal components to explain 95% of the variance. This reduces the 78 raw features down to a much smaller, decorrelated set, which both speeds up training and satisfies the assignment's feature-selection requirement. Both models below (Random Forest and the CNN) are trained on these same PCA-reduced features.

In [ ]:
pca = PCA(n_components=0.95, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Original feature count: {X_train_scaled.shape[1]}")
print(f"PCA components kept (95% variance): {pca.n_components_}")
print(f"Cumulative explained variance: {pca.explained_variance_ratio_.sum():.4f}")

plt.figure(figsize=(6, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.axhline(0.95, color="red", linestyle="--", label="95% variance")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA Explained Variance")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "pca_explained_variance.png"), dpi=150)
plt.show()

joblib.dump(scaler, os.path.join(MODELS_DIR, "scaler.joblib"))
joblib.dump(pca, os.path.join(MODELS_DIR, "pca.joblib"))
joblib.dump(label_encoder, os.path.join(MODELS_DIR, "label_encoder.joblib"))

del X_train_scaled, X_test_scaled
gc.collect()

## 6. Evaluation Helpers

Both models below are evaluated the same way, so the plotting functions are defined once here:

- A confusion matrix, shown both as raw counts and row-normalized (percentage of each true class), since raw counts are dominated by the huge BENIGN class.
- One-vs-rest precision-recall curves for every class, since this is a multi-class problem and a single PR curve is not defined the same way it is for binary classification.

In [ ]:
def plot_confusion_matrices(y_true, y_pred, title_prefix, filename_prefix):
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))

    plt.figure(figsize=(11, 9))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"{title_prefix} - Confusion Matrix (counts)")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"{filename_prefix}_confusion_matrix_counts.png"), dpi=150)
    plt.show()

    cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
    plt.figure(figsize=(11, 9))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"{title_prefix} - Confusion Matrix (row-normalized)")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"{filename_prefix}_confusion_matrix_normalized.png"), dpi=150)
    plt.show()


def plot_precision_recall_curves(y_true, y_proba, title, filename):
    y_true_bin = label_binarize(y_true, classes=range(NUM_CLASSES))

    plt.figure(figsize=(9, 7))
    avg_precisions = {}
    for i, class_name in enumerate(CLASS_NAMES):
        if y_true_bin[:, i].sum() == 0:
            continue
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_proba[:, i])
        ap = average_precision_score(y_true_bin[:, i], y_proba[:, i])
        avg_precisions[class_name] = ap
        plt.plot(recall, precision, label=f"{class_name} (AP={ap:.2f})")

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title)
    plt.legend(loc="lower left", bbox_to_anchor=(1.02, 0), fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, filename), dpi=150, bbox_inches="tight")
    plt.show()

    return avg_precisions

## 7. Classic Machine Learning - Random Forest

A `RandomForestClassifier` is trained on the PCA-reduced features for multi-class attack classification. `class_weight="balanced"` is used to partially compensate for the severe class imbalance (without it, the model could reach high accuracy by mostly ignoring rare attack classes).

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)
rf_model.fit(X_train_pca, y_train)

y_pred_rf = rf_model.predict(X_test_pca)
y_proba_rf = rf_model.predict_proba(X_test_pca)

print("=== Random Forest (PCA features) ===")
print(classification_report(y_test, y_pred_rf, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
plot_confusion_matrices(y_test, y_pred_rf, "Random Forest", "rf")
ap_rf = plot_precision_recall_curves(
    y_test, y_proba_rf, "Random Forest - Precision-Recall Curves (one-vs-rest)", "rf_precision_recall.png"
)

joblib.dump(rf_model, os.path.join(MODELS_DIR, "random_forest.joblib"))
print("Saved Random Forest model to", MODELS_DIR)

## 8. Deep Learning - 1D CNN

The same PCA-reduced feature vector is reshaped into a `(n_components, 1)` "single-channel signal" so a 1D CNN can slide convolutional filters across it, looking for locally-correlated patterns among the principal components. The output layer is a softmax over all classes, trained with sparse categorical cross-entropy (so the integer-encoded labels do not need to be one-hot encoded, saving memory on this large dataset).

In [ ]:
N_COMPONENTS = X_train_pca.shape[1]

X_train_cnn = X_train_pca.reshape(-1, N_COMPONENTS, 1)
X_test_cnn = X_test_pca.reshape(-1, N_COMPONENTS, 1)

cnn_model = Sequential([
    Conv1D(64, kernel_size=3, activation="relu", padding="same", input_shape=(N_COMPONENTS, 1)),
    MaxPooling1D(pool_size=2),
    Conv1D(128, kernel_size=3, activation="relu", padding="same"),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation="softmax"),
])

cnn_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn_model.summary()

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history = cnn_model.fit(
    X_train_cnn, y_train,
    validation_split=0.1,
    epochs=15,
    batch_size=512,
    callbacks=[early_stop],
    verbose=1,
)

In [ ]:
def plot_training_history(history, filename):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(history.history["loss"], label="Train Loss")
    axes[0].plot(history.history["val_loss"], label="Val Loss")
    axes[0].set_title("CNN Loss over Epochs")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history.history["accuracy"], label="Train Accuracy")
    axes[1].plot(history.history["val_accuracy"], label="Val Accuracy")
    axes[1].set_title("CNN Accuracy over Epochs")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, filename), dpi=150)
    plt.show()


plot_training_history(history, "cnn_training_curves.png")

y_proba_cnn = cnn_model.predict(X_test_cnn)
y_pred_cnn = np.argmax(y_proba_cnn, axis=1)

print("=== CNN (PCA features) ===")
print(classification_report(y_test, y_pred_cnn, target_names=CLASS_NAMES, zero_division=0))

plot_confusion_matrices(y_test, y_pred_cnn, "CNN", "cnn")
ap_cnn = plot_precision_recall_curves(
    y_test, y_proba_cnn, "CNN - Precision-Recall Curves (one-vs-rest)", "cnn_precision_recall.png"
)

cnn_model.save(os.path.join(MODELS_DIR, "cnn_model.keras"))
print("Saved CNN model to", MODELS_DIR)

## 9. Model Comparison

Since this is a multi-class problem with severe imbalance, macro-averaged precision/recall/F1 (unweighted mean across all 15 classes) is reported alongside overall accuracy, so that performance on rare attack classes is not hidden by the huge BENIGN class. Mean average precision (mAP, averaged across the one-vs-rest PR curves above) is also included.

In [ ]:
results = pd.DataFrame([
    {
        "Model": "Random Forest (PCA)",
        "Accuracy": accuracy_score(y_test, y_pred_rf),
        "Macro Precision": precision_score(y_test, y_pred_rf, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_test, y_pred_rf, average="macro", zero_division=0),
        "Macro F1-score": f1_score(y_test, y_pred_rf, average="macro", zero_division=0),
        "Mean Average Precision": np.mean(list(ap_rf.values())),
    },
    {
        "Model": "CNN (PCA)",
        "Accuracy": accuracy_score(y_test, y_pred_cnn),
        "Macro Precision": precision_score(y_test, y_pred_cnn, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_test, y_pred_cnn, average="macro", zero_division=0),
        "Macro F1-score": f1_score(y_test, y_pred_cnn, average="macro", zero_division=0),
        "Mean Average Precision": np.mean(list(ap_cnn.values())),
    },
]).set_index("Model")

display(results)

results.plot(kind="bar", figsize=(10, 5), ylim=(0, 1))
plt.title("Model Comparison: Classic ML vs Deep Learning")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "model_comparison.png"), dpi=150)
plt.show()

results.to_csv(os.path.join(OUTPUT_DIR, "model_comparison.csv"))
print("Saved comparison table to", os.path.join(OUTPUT_DIR, "model_comparison.csv"))

## 10. Discussion

**Findings.** Fill this in after running the notebook with the actual numbers from the comparison table above - e.g. which model achieved higher macro-F1 and mean average precision, and whether the deep learning model's extra complexity translated into a meaningful gain over the Random Forest baseline, especially on the rare attack classes.

**Cybersecurity implications.** In network intrusion detection, missing a real attack (a false negative on any attack class) is generally far more costly than a false alarm on benign traffic, since a missed DDoS, brute-force, or infiltration attempt can lead directly to a breach. This is why macro-averaged metrics and per-class precision-recall curves matter more here than overall accuracy: a model could reach over 90% accuracy just by predicting BENIGN for almost everything, since BENIGN alone makes up about 80% of the data, while still missing most real attacks. Recall on each attack class, not overall accuracy, is the metric that should drive any real deployment decision.

**Limitations.**
- Severe class imbalance is extreme for some classes: Heartbleed (11 samples), Web Attack - Sql Injection (21 samples), and Infiltration (36 samples) have so few examples in the entire dataset that their test-set metrics are based on only a handful of predictions and should not be treated as statistically reliable, even though they are reported for completeness.
- `class_weight="balanced"` in the Random Forest only partially compensates for imbalance; it does not manufacture more examples of rare classes. Techniques like SMOTE oversampling or class-specific cost-sensitive loss functions were not applied here.
- PCA reduces dimensionality but sacrifices direct interpretability: the principal components are linear combinations of the original 78 flow features, not any single named feature, which makes it harder to say which specific traffic characteristic drove a prediction.
- No hyperparameter tuning (e.g. grid search over Random Forest depth/estimators, or CNN filter sizes/learning rate) was performed; both models use reasonable defaults.
- The CNN's convolutional structure assumes some local relationship between adjacent PCA components, which is not guaranteed to be meaningful the way it would be for genuinely sequential or spatial data; an RNN over true time-ordered flow sequences, or a plain dense network, are both reasonable alternative deep learning choices for this tabular data.

**Potential improvements.**
- Apply oversampling (SMOTE) or undersampling of the BENIGN class to rebalance the training set before fitting either model.
- Report per-class recall explicitly for the rare classes in the write-up, rather than relying on macro-averages alone, to be transparent about where each model actually fails.
- Try a plain feed-forward (dense) deep network or an RNN/LSTM over engineered time windows as an alternative deep learning architecture, and compare against the CNN.
- Tune the PCA variance threshold (e.g. compare 90%, 95%, 99%) to see how much the dimensionality reduction trades off accuracy against speed and interpretability.